# Load Data

In [0]:
filepath = "/Volumes/workspace/default/hackathon/geocoded_dataset.csv"

df = spark.read \
    .option("header", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("inferSchema", False) \
    .csv(filepath)

# Remove unwanted columns

In [0]:
df = df.drop(
    "mongo DB",
    "unique_id",
    "content_table_id",
    "logo",
    "facebookLink",
    "twitterLink",
    "linkedinLink",
    "instagramLink",
    "missionStatementLink"
)

# Resolve Duplicate rows by aggregration

#### For each pk_unique_id:
* Union all useful text fields → (specialties, equipment, capability, procedure)
* Pick longest text → (description)
* Pick max numeric values → (capacity, numberDoctors)
* Pick first non-null for stable fields → (name, location, etc.)

In [0]:
# Step 1: Normalize empty strings → NULL
from pyspark.sql.functions import col, when, max, collect_set, concat_ws, expr

df = df.select([
    when(col(c) == "", None).otherwise(col(c)).alias(c)
    for c in df.columns
])

# Step 2: Merge duplicates by pk_unique_id
df_merged = df.groupBy("pk_unique_id").agg(
    # Best (longest) text fields
    expr("max_by(name, length(name))").alias("name"),
    expr("max_by(source_url, length(source_url))").alias("source_url"),
    expr("max_by(address_city, length(address_city))").alias("address_city"),
    expr("max_by(address_stateOrRegion, length(address_stateOrRegion))").alias("address_stateOrRegion"),
    expr("max_by(address_country, length(address_country))").alias("address_country"),
    expr("max_by(address_line1, length(address_line1))").alias("address_line1"),
    expr("max_by(address_line2, length(address_line2))").alias("address_line2"),
    expr("max_by(address_line3, length(address_line3))").alias("address_line3"),
    expr("max_by(address_zipOrPostcode, length(address_zipOrPostcode))").alias("address_zipOrPostcode"),
    expr("max_by(address_countryCode, length(address_countryCode))").alias("address_countryCode"),
    expr("max_by(description, length(description))").alias("description"),
    expr("max_by(missionStatement, length(missionStatement))").alias("missionStatement"),
    expr("max_by(organizationDescription, length(organizationDescription))").alias("organizationDescription"),
    expr("max_by(officialWebsite, length(officialWebsite))").alias("officialWebsite"),
    expr("max_by(location_query, length(location_query))").alias("location_query"),
    expr("max_by(osm_display_name, length(osm_display_name))").alias("osm_display_name"),
    expr("max_by(method_used_for_geocoding, length(method_used_for_geocoding))").alias("method_used_for_geocoding"),
    expr("max_by(resolved_location_query, length(resolved_location_query))").alias("resolved_location_query"),

    # Coordinates: pick max non-null value
    max("lat").alias("lat"),
    max("lon").alias("lon"),

    # List-like fields: merge unique values
    concat_ws(", ", collect_set("specialties")).alias("specialties"),
    concat_ws(", ", collect_set("procedure")).alias("procedure"),
    concat_ws(", ", collect_set("equipment")).alias("equipment"),
    concat_ws(", ", collect_set("capability")).alias("capability"),
    concat_ws(", ", collect_set("organization_type")).alias("organization_type"),
    concat_ws(", ", collect_set("phone_numbers")).alias("phone_numbers"),
    concat_ws(", ", collect_set("email")).alias("email"),
    concat_ws(", ", collect_set("websites")).alias("websites"),
    concat_ws(", ", collect_set("countries")).alias("countries"),
    concat_ws(", ", collect_set("affiliationTypeIds")).alias("affiliationTypeIds"),

    # Numeric: take max
    max("capacity").alias("capacity"),
    max("numberDoctors").alias("numberDoctors"),
    max("yearEstablished").alias("yearEstablished"),
    max("area").alias("area"),

    # IDs / categorical: take max (or use max_by with a text field if you prefer)
    max("facilityTypeId").alias("facilityTypeId"),
    max("operatorTypeId").alias("operatorTypeId"),
    max("acceptsVolunteers").alias("acceptsVolunteers"),
)

# Fix data types

In [0]:
from pyspark.sql.functions import col

df_merged = df_merged.withColumn("lat", col("lat").cast("double")) \
       .withColumn("pk_unique_id", col("pk_unique_id").cast("integer"))\
       .withColumn("lon", col("lon").cast("double")) \
       .withColumn("numberDoctors", col("numberDoctors").cast("double")) \
       .withColumn("capacity", col("capacity").cast("double")) \
       .withColumn("yearEstablished", col("yearEstablished").cast("double")) \
       .withColumn("acceptsVolunteers", col("acceptsVolunteers").cast("boolean"))

## Reorder table

In [0]:
df_merged = df_merged.select(
    "pk_unique_id",
    "name",
    "source_url",
    "specialties",
    "procedure",
    "equipment",
    "capability",
    "organization_type",
    "phone_numbers",
    "email",
    "websites",
    "officialWebsite",
    "yearEstablished",

    "acceptsVolunteers",
    "missionStatement",
    "organizationDescription",
    "facilityTypeId",
    "operatorTypeId",
    "affiliationTypeIds",
    "description",
    "area",
    "numberDoctors",
    "capacity",
    
    "address_line1",
    "address_line2",
    "address_line3",
    "address_city",
    "address_stateOrRegion",
    "address_zipOrPostcode",
    "address_country",
    "address_countryCode",
    "countries",

    "lat",
    "lon",
    "location_query",
    "osm_display_name",
    "method_used_for_geocoding",
    "resolved_location_query"
)

# Save Table

In [0]:
df_merged = df_merged.orderBy("pk_unique_id")
df_merged.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.facilities2")